In [7]:
"""
Script para treinar o modelo final com todos os dados e salvá-lo para produção.
"""
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
import logging

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def train_final_model():
    """
    Treina o modelo Random Forest com todos os dados e salva:
    1. O modelo treinado (.pkl)
    2. A lista de features usadas (.pkl)
    """
    
    # 1. Carregar dados processados
    logging.info("Carregando dados de treino...")
    df = pd.read_csv('../../data/features_train.csv')
    
    # Separar features e target
    X = df.drop(columns=['target'])
    y = df['target']
    
    # Salvar a lista de features para uso posterior (importante!)
    feature_names = X.columns.tolist()
    
    logging.info(f"Dados carregados: {X.shape}")
    logging.info(f"Features: {feature_names}")
    logging.info(f"Distribuição target: {y.value_counts().to_dict()}")
    
    # 2. Treinar modelo final com todos os dados
    logging.info("Treinando Random Forest com todos os dados...")
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,  # usar todos os cores
        verbose=1
    )
    
    model.fit(X, y)
    
    # 3. Avaliar performance no próprio treino (apenas para referência)
    y_pred_proba = model.predict_proba(X)[:, 1]
    train_logloss = -np.mean(y * np.log(y_pred_proba + 1e-15) + (1-y) * np.log(1 - y_pred_proba + 1e-15))
    
    logging.info(f"LogLoss no treino: {train_logloss:.6f}")
    
    # 4. Salvar modelo e metadados
    logging.info("Salvando modelo...")
    
    # Criar diretório models se não existir
    os.makedirs('models', exist_ok=True)
    
    # Salvar modelo
    model_path = 'models/random_forest_final.pkl'
    joblib.dump(model, model_path)
    
    # Salvar lista de features (importante para garantir mesma ordem no teste)
    features_path = 'models/feature_names.pkl'
    joblib.dump(feature_names, features_path)
    
    # Salvar também em formato txt para fácil visualização
    with open('models/feature_names.txt', 'w') as f:
        f.write('\n'.join(feature_names))
    
    logging.info(f"Modelo salvo em: {model_path}")
    logging.info(f"Features salvas em: {features_path}")
    
    # 5. Importância das features
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    logging.info("\nTop 10 features mais importantes:")
    for i in range(10):
        logging.info(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")
    
    return model, feature_names

if __name__ == "__main__":
    train_final_model()

2026-02-26 18:08:31,495 - INFO - Carregando dados de treino...
2026-02-26 18:08:31,563 - INFO - Dados carregados: (33474, 25)
2026-02-26 18:08:31,564 - INFO - Features: ['length', 'n_keydown', 'n_keyup', 'n_unique_codes', 'total_time', 'first_tick', 'hold_mean', 'hold_std', 'hold_min', 'hold_max', 'hold_median', 'hold_q25', 'hold_q75', 'hold_cv', 'flight_mean', 'flight_std', 'flight_min', 'flight_max', 'flight_median', 'flight_cv', 'press_press_mean', 'press_press_std', 'press_press_cv', 'keys_per_second', 'hold_flight_ratio']
2026-02-26 18:08:31,565 - INFO - Distribuição target: {1: 17245, 0: 16229}
2026-02-26 18:08:31,566 - INFO - Treinando Random Forest com todos os dados...
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs